# Gate simples R1 — precisão e coverage

Este notebook testa somente os três critérios definidos para esta etapa:

- precisão de improdutividade >= 85%;
- precisão de produtividade >= 85%;
- coverage >= 65% (ideal >= 75%).

R1 envia `acao_indefinida`/`nao_nomeado` para validação. R4 acrescenta veto por vizinho produtivo apenas para comparar o impacto de combinar duas regras.

In [ ]:
from pathlib import Path
import json, os, sys
import pandas as pd

ROOT = Path.cwd()
NB_DIR = ROOT / 'notebooks' if (ROOT / 'notebooks').is_dir() else ROOT
REPO = NB_DIR.parent
sys.path.insert(0, str(NB_DIR))
sys.path.insert(0, str(REPO))

from produtividade_30d import A, aplicar_regras, carregar_fontes, dividir_por_dia, metricas, preparar_dataset
from backend.productivity import acao_indefinida_deve_abster

OUT = Path(os.environ.get('KV_SIMPLE_GATE_OUTPUT_DIR', NB_DIR / 'outputs' / 'productivity_simple_gate'))
OUT.mkdir(parents=True, exist_ok=True)
eventos, catalogo, fontes = carregar_fontes()
proxy, _ = preparar_dataset(eventos, catalogo)
split = dividir_por_dia(proxy)
print('Fontes:', *(str(p) for p in fontes), sep='\n- ')

In [ ]:
def predicao_r1(df):
    pred = df['y_baseline'].copy()
    # O replay reproduz a decisão na ingestão: a correção posterior é a
    # verdade de comparação, nunca uma entrada da previsão.
    mascara = df.apply(lambda r: acao_indefinida_deve_abster({
        'comportamento_label': r.get('comportamento_label'),
    }), axis=1)
    pred[mascara] = A
    return pred

def resumo(parte, politica, df, pred):
    m = metricas(df, pred)
    return {
        'parte': parte,
        'politica': politica,
        'precisao_improdutividade_pct': round(100 * m['precision_I'], 2),
        'precisao_produtividade_pct': round(100 * m['precision_P'], 2),
        'coverage_pct': round(100 * m['coverage'], 2),
    }

linhas = []
for parte, df in [('calibracao', split.calibracao), ('teste_interno', split.teste_interno)]:
    regras = aplicar_regras(df)
    linhas.extend([
        resumo(parte, 'R0_baseline', df, df['y_baseline']),
        resumo(parte, 'R1_indefinida_abstem', df, predicao_r1(df)),
        resumo(parte, 'R4_R1_mais_veto_vizinho', df, regras['R4_indefinida_mais_veto']),
    ])
resultado = pd.DataFrame(linhas)
resultado

In [ ]:
resultado['passa_gate'] = (
    (resultado.precisao_improdutividade_pct >= 85)
    & (resultado.precisao_produtividade_pct >= 85)
    & (resultado.coverage_pct >= 65)
)
resultado['coverage_ideal'] = resultado.coverage_pct >= 75
r1 = resultado[resultado.politica == 'R1_indefinida_abstem']
assert r1.passa_gate.all(), resultado

decisao = {
    'regra_escolhida': 'R1_indefinida_abstem',
    'motivo': 'passa o gate nas duas partes e preserva mais coverage que R4',
    'gate': {'precisao_I_min_pct': 85, 'precisao_P_min_pct': 85, 'coverage_min_pct': 65, 'coverage_ideal_pct': 75},
    'resultados': r1.to_dict(orient='records'),
}
resultado.to_csv(OUT / 'resultado_gate_simples.csv', index=False)
(OUT / 'decisao_gate_simples.json').write_text(json.dumps(decisao, indent=2, ensure_ascii=False), encoding='utf-8')
print(json.dumps(decisao, indent=2, ensure_ascii=False))
resultado